# Broadband Continuous Signal Processing with STFT-Based Propagation

This notebook demonstrates broadband passive-sonar signal generation and reception using the `BroadbandPassiveSonarArraySimulator`. It follows a single moving target and a towed array through source generation, frequency-domain propagation, array reception, and signal analysis.

**Background**
- Broadband ship noise is not well represented by a handful of discrete tones; realistic signals span a wider band and evolve continuously in time.
- For moving-source, moving-array problems, propagation must preserve phase and timing well enough to reconstruct received sensor data after frequency-domain processing.
- In passive sonar analysis, spectrograms and spectra are only interpretable if FFT scaling and PSD conventions are handled correctly.

**Key Concepts**
- STFT-based frequency-domain propagation and overlap-add reconstruction.
- Broadband source modelling with time-varying transfer functions.
- Correct amplitude interpretation in FFT and spectrogram-based plots.


## Setup and Reproducibility


In [1]:
from datetime import datetime, timedelta

import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from stonesoup.models.transition.linear import (
    CombinedLinearGaussianTransitionModel,
    ConstantVelocity,
)
from stonesoup.types.groundtruth import GroundTruthPath, GroundTruthState

from nereus.models.environment import FlatBathymetry, Linear
from nereus.models.propagation import rtrsAcousticPropagationModel
from nereus.platform import TowedArrayPlatform
from nereus.plotter import plot_spectrogram, plot_world
from nereus.signal.anthropogenic import BroadbandShipSignal
from nereus.simulator import BroadbandPassiveSonarArraySimulator

# Set random seed for reproducibility
np.random.seed(1999)

## Simulation Parameters

Configure the simulation timing: 60 seconds total duration with 2-second timesteps (30 steps).

In [2]:
# Simulation parameters
SIM_RATE = 2.0
SIM_PARAMS = {
    "start_time": datetime.now().replace(hour=0, minute=0, second=0, microsecond=0),
    "time_interval": timedelta(seconds=SIM_RATE),
    "num_steps": 30,
}

total_duration_s = SIM_PARAMS["num_steps"] * SIM_PARAMS["time_interval"].total_seconds()

print("=== Broadband Continuous Signal Processing Demo ===")
print(f"Total simulation duration: {total_duration_s} s")
print(f"Number of timesteps: {SIM_PARAMS['num_steps']}")
print(f"Timestep interval: {SIM_PARAMS['time_interval'].total_seconds()} s")

=== Broadband Continuous Signal Processing Demo ===
Total simulation duration: 60.0 s
Number of timesteps: 30
Timestep interval: 2.0 s


## Platform and Array Parameters

The ship platform remains stationary while the towed array (100 sensors, 200m cable, 1m spacing) is deployed at 200m depth.

In [3]:
SHIP_PARAMS = {
    "start_vector": np.array([0, 0, 0, 0, -10.0, 0]),  # Stationary ship
    "position_mapping": [0, 2, 4],
    "velocity_mapping": [1, 3, 5],
    "transition_model": CombinedLinearGaussianTransitionModel(
        [ConstantVelocity(0), ConstantVelocity(0), ConstantVelocity(0)]
    ),
}

ARRAY_PARAMS = {
    "num_sensors": 100,
    "tow_cable_length": 200.0,
    "sensor_spacing": 1.0,
    "array_depth": -200.0,
}

print("Platform: Stationary ship at surface")
print(
    f"Array: {ARRAY_PARAMS['num_sensors']} sensors, "
    f"{ARRAY_PARAMS['tow_cable_length']}m cable, "
    f"{ARRAY_PARAMS['sensor_spacing']}m spacing"
)
print(f"Array depth: {ARRAY_PARAMS['array_depth']} m")

Platform: Stationary ship at surface
Array: 100 sensors, 200.0m cable, 1.0m spacing
Array depth: -200.0 m


## Target Parameters

The target starts at 4.5 km range with slow motion. Its acoustic signature consists of 4 tonal components at different frequencies (60, 85, 120, 200 Hz) with amplitudes ranging from 75-100 dB re 1 µPa.

In [4]:
TARGET_PARAMS = {
    "start_vector": np.array([0, 0, 4500, -10, -10.0, 0]),  # Moving target
    "position_mapping": [0, 2, 4],
    "velocity_mapping": [1, 3, 5],
    "transition_model": CombinedLinearGaussianTransitionModel(
        [ConstantVelocity(0.001), ConstantVelocity(0.001), ConstantVelocity(0)]
    ),
    "amplitudes_upa": 10 ** (np.array([80.0, 95.0, 100.0, 75.0]) / 20),
    "frequencies_hz": np.array([60.0, 85.0, 120.0, 200.0]),
    "phases_rad": np.random.uniform(0, 2 * np.pi, 4),
    "tonal_bandwidth_hz": 2.0,
    "noise_amplitude_upa": 10 ** (80 / 20),
    "noise_spectral_exponent": -1.0,
}

# Convert amplitudes to dB for display
amplitudes_db = 20 * np.log10(TARGET_PARAMS["amplitudes_upa"])

print(f"Target initial position: {TARGET_PARAMS['start_vector'][2]:.0f} m range")
print(f"Target depth: {TARGET_PARAMS['start_vector'][4]:.0f} m")
print(f"Tonal frequencies: {TARGET_PARAMS['frequencies_hz']} Hz")
print(f"Tonal amplitudes: {amplitudes_db} dB re 1 µPa")

Target initial position: 4500 m range
Target depth: -10 m
Tonal frequencies: [ 60.  85. 120. 200.] Hz
Tonal amplitudes: [ 80.  95. 100.  75.] dB re 1 µPa


## Signal and Propagation Parameters

**Signal Parameters**: STFT processing with 500-sample frames (1 second at 500 Hz sampling rate) and 75% overlap (hop_factor=4).

**Propagation Parameters**: Constant sound speed profile and flat bathymetry at 100m depth.

In [5]:
SIGNAL_PARAMS = {
    "duration_s": total_duration_s,  # Long continuous signal
    "sampling_rate_hz": 500.0,
    "frame_len": 500,  # STFT frame length
    "hop_factor": 2,  # 75% overlap
    "fade_in_ms": 1000.0,  # Gentle fade-in
}

PROP_PARAMS = {
    "ssp": Linear(surface_speed=1500.0, gradient=0.2),
    "attenuation_factor": 0.5,
    "bathymetry": FlatBathymetry(depth=-150.0),
    "step_m": 20.0,
    "azimuth_search_width": 2.0,
    "azimuth_resolution": 0.5,
    "elevation_range": (-25.0, 25.0),
    "elevation_resolution": 1.0,
}

# Sensor to analyze
SENSOR_TO_ANALYZE = ARRAY_PARAMS["num_sensors"] // 2

## Platform Generation

Initialize the towed array platform and simulate motion through timesteps.

In [6]:
print("Creating platform with vertical motion...")
initial_state = GroundTruthState(SHIP_PARAMS["start_vector"], timestamp=SIM_PARAMS["start_time"])

platform = TowedArrayPlatform(
    states=[initial_state],
    position_mapping=SHIP_PARAMS["position_mapping"],
    velocity_mapping=SHIP_PARAMS["velocity_mapping"],
    transition_models=[SHIP_PARAMS["transition_model"]],
    transition_times=[timedelta(seconds=total_duration_s)],
    num_sensors=ARRAY_PARAMS["num_sensors"],
    cable_length_m=ARRAY_PARAMS["tow_cable_length"],
    sensor_spacing_m=ARRAY_PARAMS["sensor_spacing"],
    array_depth_m=ARRAY_PARAMS["array_depth"],
)

# Add vertical motion to platform
depth_change_per_step = -10.0  # Move up 10m per timestep
for i in range(1, SIM_PARAMS["num_steps"]):
    new_time = SIM_PARAMS["start_time"] + i * SIM_PARAMS["time_interval"]
    platform.move(new_time)

# Get final array depth
final_platform_state = platform.get_platform_state_at(
    SIM_PARAMS["start_time"] + (SIM_PARAMS["num_steps"] - 1) * SIM_PARAMS["time_interval"]
)
final_depth = (
    final_platform_state.array.state_vector[2, 0]
    if final_platform_state
    else ARRAY_PARAMS["array_depth"]
)

print(f"  Array depth: {ARRAY_PARAMS['array_depth']} m -> {final_depth:.1f} m")

Creating platform with vertical motion...
  Array depth: -200.0 m -> -200.0 m


## Target Trajectory

Generate the target ground truth path with acoustic metadata.

In [7]:
target_states = [
    GroundTruthState(
        TARGET_PARAMS["start_vector"],
        timestamp=SIM_PARAMS["start_time"],
        metadata={
            "amplitudes_upa": TARGET_PARAMS["amplitudes_upa"],
            "frequencies_hz": TARGET_PARAMS["frequencies_hz"],
            "phases_rad": TARGET_PARAMS["phases_rad"],
            "position_mapping": TARGET_PARAMS["position_mapping"],
            "velocity_mapping": TARGET_PARAMS["velocity_mapping"],
        },
    )
]

transition_model = TARGET_PARAMS["transition_model"]
for i in range(1, SIM_PARAMS["num_steps"]):
    new_time = SIM_PARAMS["start_time"] + i * SIM_PARAMS["time_interval"]
    time_interval = new_time - target_states[-1].timestamp
    new_state_vector = transition_model.function(
        target_states[-1], noise=False, time_interval=time_interval
    )
    new_state = GroundTruthState(
        new_state_vector,
        timestamp=new_time,
        metadata=target_states[-1].metadata,
    )
    target_states.append(new_state)

target_ground_truth = GroundTruthPath(target_states)

## Broadband Signal Model

Use **BroadbandShipSignal** for realistic ship acoustics: broadband tonals (1 Hz bandwidth to simulate mechanical resonances) plus colored noise (85 dB re 1 µPa, pink noise spectrum with 1/f decay).

In [8]:
signal_model = BroadbandShipSignal(
    duration_s=SIGNAL_PARAMS["duration_s"],
    sampling_rate_hz=SIGNAL_PARAMS["sampling_rate_hz"],
    frame_len=SIGNAL_PARAMS["frame_len"],
    hop_factor=SIGNAL_PARAMS["hop_factor"],
    tonal_bandwidth_hz=1.0,  # Broader tonals (simulates mechanical resonances)
    noise_amplitude_upa=10 ** (85 / 20),  # 85 dB re 1 µPa background noise
    noise_spectral_exponent=-1.0,  # Pink noise (1/f)
    noise_freq_range_hz=(0.0, SIGNAL_PARAMS["sampling_rate_hz"]),
)

## Propagation Model and Simulator

Set up the **rtrs** (Range-dependent Two-dimensional Ray Simulation) acoustic propagation model and initialize the broadband passive sonar simulator.

In [9]:
prop_model = rtrsAcousticPropagationModel(
    ssp=PROP_PARAMS["ssp"],
    bathymetry=PROP_PARAMS["bathymetry"],
    step_m=PROP_PARAMS["step_m"],
    azimuth_search_width=PROP_PARAMS["azimuth_search_width"],
    azimuth_resolution=PROP_PARAMS["azimuth_resolution"],
    elevation_range=PROP_PARAMS["elevation_range"],
    elevation_resolution=PROP_PARAMS["elevation_resolution"],
)

signal_model = BroadbandShipSignal(
    duration_s=SIGNAL_PARAMS["duration_s"],
    sampling_rate_hz=SIGNAL_PARAMS["sampling_rate_hz"],
    frame_len=SIGNAL_PARAMS["frame_len"],
    hop_factor=SIGNAL_PARAMS["hop_factor"],
    tonal_bandwidth_hz=TARGET_PARAMS["tonal_bandwidth_hz"],
    noise_amplitude_upa=TARGET_PARAMS["noise_amplitude_upa"],
    noise_spectral_exponent=TARGET_PARAMS["noise_spectral_exponent"],
    noise_freq_range_hz=(0.0, SIGNAL_PARAMS["sampling_rate_hz"] / 2),
    tonal_noise_is_constant=True,
    noise_is_constant=True,
)

simulator = BroadbandPassiveSonarArraySimulator(
    platform=platform,
    propagation_model=prop_model,
    signal_models=[signal_model],
    noise_model=None,
    beamformer=None,
    steering_calculator=None,
    ground_truth_paths=[target_ground_truth],
    fade_in_ms=SIGNAL_PARAMS["fade_in_ms"],
)

## Run Simulation and Collect Sensor Data

Execute the simulation loop to generate sensor data at each timestep. The simulator processes signals in the frequency domain using STFT, applies propagation transfer functions, and reconstructs time-domain signals via overlap-add.

**Note**: This step may take several minutes depending on system performance.

In [10]:
all_sensor_signals = []

for _, sensor_data_set in simulator.sensor_data_gen():
    sensor_data = next(iter(sensor_data_set))
    all_sensor_signals.append(sensor_data.raw_signals)

# Concatenate all timestep signals
all_sensor_signals_array = np.concatenate(all_sensor_signals, axis=1)
continuous_signal = all_sensor_signals_array[SENSOR_TO_ANALYZE, :]
continuous_signal_first = all_sensor_signals_array[0, :]
continuous_signal_last = all_sensor_signals_array[-1, :]
time_axis = np.arange(continuous_signal.shape[0]) / SIGNAL_PARAMS["sampling_rate_hz"]

print(f"Total signal length: {len(continuous_signal)} samples")
print(f"Signal duration: {len(continuous_signal) / SIGNAL_PARAMS['sampling_rate_hz']:.1f} s")


Total signal length: 29500 samples
Signal duration: 59.0 s


## Retrieve Source Signal

Extract the source signal from the signal model cache

In [11]:
try:
    source_signal = signal_model.get_source_signal()
except RuntimeError:
    print("Computing STFT to generate source signal...")
    signal_model.compute_stft(target_states[0])
    source_signal = signal_model.get_source_signal()

## Time-Domain Signal and Spectrogram

Plot the continuous time-domain signal and its spectrogram. Red dashed lines mark timestep boundaries where the propagation transfer function is updated.

**Important**: The spectrogram uses proper PSD scaling (dB re 1 µPa²/Hz) for underwater acoustics.

In [12]:
boundary_times_s = [
    i * SIM_PARAMS["time_interval"].total_seconds()
    for i in range(1, SIM_PARAMS["num_steps"])
]
signal_real = np.real(continuous_signal)

# Plot 1: Time-domain signal
fig_time = go.Figure()
fig_time.add_trace(
    go.Scatter(x=time_axis, y=signal_real, mode="lines", line=dict(width=1), opacity=0.7)
)
fig_time.update_yaxes(
    range=[np.percentile(signal_real, 1), np.percentile(signal_real, 99)],
    title_text="Pressure (µPa)",
)
fig_time.update_xaxes(title_text="Time (s)")

for t_boundary in boundary_times_s:
    fig_time.add_vline(
        x=t_boundary,
        line_color="black",
        line_dash="dash",
        opacity=0.5,
        line_width=1,
    )

fig_time.update_layout(
    title=f"Continuous Time-Domain Signal - Sensor {SENSOR_TO_ANALYZE}",
    template="plotly_white",
    height=400,
    width=900,
)
fig_time.show()

# Plot 2: Spectrogram (use plotter helper)
fig_spec = plot_spectrogram(
    signal_real,
    int(SIGNAL_PARAMS["sampling_rate_hz"]),
    n_fft=256,
    hop_length=64,
    y_lim=(0, SIGNAL_PARAMS["sampling_rate_hz"] / 2),
    yaxis_format="hz",
    figsize=(9, 4),
)
for t_boundary in boundary_times_s:
    fig_spec.add_vline(
        x=t_boundary,
        line_color="grey",
        line_dash="dash",
        opacity=0.5,
        line_width=1,
    )
fig_spec.update_layout(title="Spectrogram - Broadband Continuous Signal")
fig_spec.show()

## Target and Platform Trajectories

Show the trajectories in 2D Cartesian space (top view).

Note: The platform is shown as a single point because it does not move, but the array is in fact
perpendicular to the target trajectory.

In [13]:
# Use plotter helper for world view
fig_world = plot_world(truths=[target_ground_truth], platform=platform)
fig_world.update_layout(title="World Picture")
fig_world.show()

## First and Last Sensor Comparison

Visualise spatial variation: signals from opposite ends of the 100-sensor array show time delays and amplitude differences due to array geometry.

In [14]:
boundary_times_s = [
    i * SIM_PARAMS["time_interval"].total_seconds()
    for i in range(1, SIM_PARAMS["num_steps"])
]
first_real = np.real(continuous_signal_first)
last_real = np.real(continuous_signal_last)

fig_end_sensors = make_subplots(
    rows=2,
    cols=1,
    shared_xaxes=True,
    vertical_spacing=0.1,
    subplot_titles=("Sensor 0", f"Sensor {ARRAY_PARAMS['num_sensors'] - 1}"),
)

# First sensor
fig_end_sensors.add_trace(
    go.Scatter(
        x=time_axis,
        y=first_real,
        mode="lines",
        line=dict(width=1, color="steelblue"),
        opacity=0.8,
        showlegend=False,
    ),
    row=1,
    col=1,
)
fig_end_sensors.update_yaxes(
    range=[np.percentile(first_real, 1), np.percentile(first_real, 99)],
    title_text="Pressure (µPa)",
    row=1,
    col=1,
)

# Last sensor
fig_end_sensors.add_trace(
    go.Scatter(
        x=time_axis,
        y=last_real,
        mode="lines",
        line=dict(width=1, color="darkorange"),
        opacity=0.8,
        showlegend=False,
    ),
    row=2,
    col=1,
)
fig_end_sensors.update_yaxes(
    range=[np.percentile(last_real, 1), np.percentile(last_real, 99)],
    title_text="Pressure (µPa)",
    row=2,
    col=1,
)
fig_end_sensors.update_xaxes(title_text="Time (s)", row=2, col=1)

for row in (1, 2):
    for t_boundary in boundary_times_s:
        fig_end_sensors.add_vline(
            x=t_boundary,
            line_color="black",
            line_dash="dash",
            opacity=0.5,
            line_width=1,
            row=row,
            col=1,
        )

fig_end_sensors.update_layout(
    title="Continuous Time-Domain Signals - Array End Sensors",
    template="plotly_white",
    height=760,
    width=900,
)
fig_end_sensors.show()


## Source Signal Analysis

Examine the source signal before propagation:
- **Time domain**: Shows the temporal structure
- **Spectrogram**: Reveals frequency content evolution over time
- **Frequency spectrum**: Shows tonal peaks with proper amplitude scaling

**Critical FFT Scaling**: We multiply by `2.0/N` to convert FFT magnitude to single-sided amplitude spectrum in µPa.

In [15]:
source_time_axis = np.arange(len(source_signal)) / SIGNAL_PARAMS["sampling_rate_hz"]
source_real = np.real(source_signal)

# Top: Source time domain
fig_src_time = go.Figure()
fig_src_time.add_trace(
    go.Scatter(
        x=source_time_axis,
        y=source_real,
        mode="lines",
        line=dict(width=1, color="steelblue"),
        opacity=0.8,
    )
)
fig_src_time.update_layout(
    title="Source Signal - Time Domain",
    xaxis_title="Time (s)",
    yaxis_title="Amplitude (µPa)",
    template="plotly_white",
    height=400,
    width=900,
)
fig_src_time.show()

# Bottom: Source spectrogram
fig_src_spec = plot_spectrogram(
    source_real,
    int(SIGNAL_PARAMS["sampling_rate_hz"]),
    n_fft=256,
    hop_length=64,
    y_lim=(0, SIGNAL_PARAMS["sampling_rate_hz"] / 2),
    yaxis_format="hz",
    figsize=(9, 4),
)
fig_src_spec.update_layout(title="Source Signal - Spectrogram")
fig_src_spec.show()

## Source Frequency Spectrum

Plot the frequency spectrum of the source signal with **proper amplitude scaling**:
- FFT magnitude is multiplied by `2.0/N` to get single-sided amplitude
- Converted to dB re 1 µPa: `20 * log10(amplitude)`
- Red dashed lines mark target frequencies (60, 85, 120, 200 Hz)

In [16]:
signal_real_src = np.real(source_signal)
freq_spectrum_src = np.fft.rfft(signal_real_src)
freq_axis_src = np.fft.rfftfreq(len(signal_real_src), 1 / SIGNAL_PARAMS["sampling_rate_hz"])

# FFT scaling: multiply by 2/N to get amplitude
N = len(signal_real_src)
spectrum_amplitude_upa = np.abs(freq_spectrum_src) * (2.0 / N)
spectrum_magnitude_db_src = 20 * np.log10(spectrum_amplitude_upa + 1e-10)

fig_src_freq = go.Figure()
fig_src_freq.add_trace(
    go.Scatter(
        x=freq_axis_src,
        y=spectrum_magnitude_db_src,
        mode="lines",
        line=dict(width=1, color="steelblue"),
        opacity=0.8
    )
)
for freq in TARGET_PARAMS["frequencies_hz"]:
    fig_src_freq.add_vline(
        x=float(freq),
        line_color="black",
        line_dash="dash",
        opacity=0.5,
        line_width=1.5,
    )

fig_src_freq.update_layout(
    title="Source Signal - Frequency Spectrum",
    xaxis_title="Frequency (Hz)",
    yaxis_title="Magnitude (dB re 1 µPa)",
    template="plotly_white",
    width=900,
    height=520,
)
fig_src_freq.show()


## Source and Received Spectral Slices

Extract 1-second spectral slices at the middle timestep (t=30s) and compare source and received signal spectra. This reveals propagation effects:

- **Transmission loss**: Reduction in amplitude at all frequencies
- **Frequency-dependent attenuation**: Higher frequencies may be attenuated more
- **Spectral distortion**: Changes in relative tonal amplitudes due to multipath interference

Both spectra use proper FFT scaling (`2.0/N` factor) for amplitude in dB re 1 µPa.

In [17]:
# Calculate middle timestep
middle_timestep_idx = SIM_PARAMS["num_steps"] // 2
timestep_duration_s = SIM_PARAMS["time_interval"].total_seconds()
middle_time_s = middle_timestep_idx * timestep_duration_s

# Extract 1-second slice around middle time
slice_duration_s = 1.0
slice_half_duration_s = slice_duration_s / 2
slice_start_time_s = middle_time_s - slice_half_duration_s
slice_end_time_s = middle_time_s + slice_half_duration_s

slice_start_sample = int(slice_start_time_s * SIGNAL_PARAMS["sampling_rate_hz"])
slice_end_sample = int(slice_end_time_s * SIGNAL_PARAMS["sampling_rate_hz"])

source_slice = source_signal[slice_start_sample:slice_end_sample]
received_slice = continuous_signal[slice_start_sample:slice_end_sample]

# Compute FFT with correct scaling
source_slice_real = np.real(source_slice)
received_slice_real = np.real(received_slice)

freq_spectrum_source_slice = np.fft.rfft(source_slice_real)
freq_spectrum_received_slice = np.fft.rfft(received_slice_real)
freq_axis_slice = np.fft.rfftfreq(len(source_slice_real), 1 / SIGNAL_PARAMS["sampling_rate_hz"])

N_slice = len(source_slice_real)
spectrum_amplitude_source_slice = np.abs(freq_spectrum_source_slice) * (2.0 / N_slice)
spectrum_amplitude_received_slice = np.abs(freq_spectrum_received_slice) * (2.0 / N_slice)

spectrum_magnitude_db_source_slice = 20 * np.log10(spectrum_amplitude_source_slice + 1e-10)
spectrum_magnitude_db_received_slice = 20 * np.log10(spectrum_amplitude_received_slice + 1e-10)

fig_slice_compare = make_subplots(
    rows=2,
    cols=1,
    shared_xaxes=True,
    vertical_spacing=0.10,
    subplot_titles=(
        f"Source Signal - Spectral Slice at t={middle_time_s:.1f}s (±{slice_half_duration_s}s)",
        f"Received Signal - Spectral Slice at t={middle_time_s:.1f}s (±{slice_half_duration_s}s)",
    ),
)

# Source slice
fig_slice_compare.add_trace(
    go.Scatter(
        x=freq_axis_slice,
        y=spectrum_magnitude_db_source_slice,
        mode="lines",
        line=dict(width=1, color="steelblue"),
        showlegend=False,
    ),
    row=1,
    col=1,
)
fig_slice_compare.update_yaxes(title_text="Magnitude (dB re 1 µPa)", row=1, col=1)
fig_slice_compare.update_xaxes(range=[0, 250], row=1, col=1)
for freq in TARGET_PARAMS["frequencies_hz"]:
    fig_slice_compare.add_vline(
        x=float(freq),
        line_color="black",
        line_dash="dash",
        opacity=0.5,
        line_width=1.5,
        row=1,
        col=1,
    )

# Received slice
fig_slice_compare.add_trace(
    go.Scatter(
        x=freq_axis_slice,
        y=spectrum_magnitude_db_received_slice,
        mode="lines",
        line=dict(width=1, color="darkorange"),
        showlegend=False,
    ),
    row=2,
    col=1,
)
fig_slice_compare.update_yaxes(title_text="Magnitude (dB re 1 µPa)", row=2, col=1)
fig_slice_compare.update_xaxes(title_text="Frequency (Hz)", range=[0, 250], row=2, col=1)
for freq in TARGET_PARAMS["frequencies_hz"]:
    fig_slice_compare.add_vline(
        x=float(freq),
        line_color="black",
        line_dash="dash",
        opacity=0.5,
        line_width=1.5,
        row=2,
        col=1,
    )

fig_slice_compare.update_layout(
    title="Spectral Slice Comparison - Source vs Received at Middle Timestep",
    template="plotly_white",
    height=760,
    width=900,
)
fig_slice_compare.show()